# 13d. SGLang & Request Scheduling

**Tier:** Building with LLMs
**Estimated time:** 50 minutes
**Prerequisites:** 13b (vLLM inference serving), 13c (KV cache eviction)
**Priority:** 🟡 Important — the two ideas here (prefix reuse across requests, principled admission control) are exactly what separates a demo server from a production one under real traffic. *If skipped, revisit when:* your workload is agentic or few-shot (long shared system prompts, many similar requests) and vLLM's per-request paging alone isn't giving you the throughput you expected — or when your server starts timing out short requests behind long ones under load.
**Source material:** Zheng et al., *"SGLang: Efficient Execution of Structured Language Model Programs"* (NeurIPS 2024) — https://arxiv.org/abs/2312.07104 (RadixAttention); Stanford LLM curriculum, Lecture 3 (inference optimization).

## What You'll Learn
- How **RadixAttention** reuses KV cache across *different requests* that share a prefix — not just within one request the way notebook 13's prompt caching does
- Why that reuse matters most for exactly the workloads this curriculum builds: agentic loops and few-shot prompts, where the same system prompt or tool schema repeats across thousands of calls
- The difference between vLLM's PagedAttention (per-request memory management, notebook 13b) and SGLang's RadixAttention (cross-request memory *sharing*) — they solve different problems and modern serving stacks increasingly do both
- How a server decides **which waiting request to run next** when it can't run them all — FCFS vs shortest-job-first vs priority — and what each policy costs you in tail latency

## Why This Matters
Notebook 13b's continuous batching answered "how do I keep the GPU full." This notebook answers two questions that come right after: "how do I avoid recomputing the same tokens over and over," and "when there are more requests than I can run at once, which one goes first." Both matter enormously for agentic and RAG workloads — a coding agent that resends the same 4,000-token system prompt and tool schema on every one of its 50 tool calls, or a support bot answering many users against the same product-docs system prompt, is paying full price for tokens it's already computed a hundred times, and a slow FCFS queue can make a two-token "yes/no" classification wait behind someone else's 2,000-token essay.


## RadixAttention: reuse KV cache *across* requests, not just within one

Notebook 13's prompt caching sped up **one conversation** by reusing the KV cache for a prefix that conversation had already computed. RadixAttention generalizes that to **every request the server has ever seen**: if two different requests — from two different users, at two different times — happen to share the same opening tokens (a system prompt, a few-shot block, a repeated tool schema), the second one can reuse the first one's cached KV for that shared span instead of recomputing it.

The data structure that makes this fast is a **prefix tree**: every token sequence the server has processed is inserted into the tree one token at a time, and a new request's cache-hit length is however far down the tree it can walk before its tokens diverge from anything seen before. (SGLang's actual RadixAttention additionally *compresses* single-child chains into single edges — the "radix" part — purely for memory efficiency; that compression doesn't change which tokens get reused, so the plain trie below has identical cache-hit behavior and is simpler to read.)


In [1]:
import numpy as np
rng = np.random.default_rng(0)

class PrefixTrie:
    """Tracks every token sequence inserted so far; insert() returns how many leading
    tokens of a new sequence were ALREADY present (reused) vs newly added (recomputed).
    """
    def __init__(self):
        self.root = {}
        self.total_tokens_seen = 0
        self.total_tokens_reused = 0

    def insert(self, token_seq: tuple) -> tuple[int, int]:
        node = self.root
        reused = 0
        i = 0
        while i < len(token_seq) and token_seq[i] in node:
            node = node[token_seq[i]]
            reused += 1
            i += 1
        new_tokens = len(token_seq) - reused
        for tok in token_seq[i:]:
            node[tok] = {}
            node = node[tok]
        self.total_tokens_seen += len(token_seq)
        self.total_tokens_reused += reused
        return reused, new_tokens

trie = PrefixTrie()
r1 = trie.insert((1, 2, 3, 4, 5))
r2 = trie.insert((1, 2, 3, 9, 9))   # shares the first 3 tokens with r1, then diverges
r3 = trie.insert((1, 2, 3, 4, 5))   # an EXACT repeat of r1 -- fully reused
print(f"Request 1 (1,2,3,4,5)   -> reused {r1[0]}, new {r1[1]}")
print(f"Request 2 (1,2,3,9,9)   -> reused {r2[0]}, new {r2[1]}  (shares the 1,2,3 prefix)")
print(f"Request 3 (1,2,3,4,5)   -> reused {r3[0]}, new {r3[1]}  (a repeat of request 1)")


Request 1 (1,2,3,4,5)   -> reused 0, new 5
Request 2 (1,2,3,9,9)   -> reused 3, new 2  (shares the 1,2,3 prefix)
Request 3 (1,2,3,4,5)   -> reused 5, new 0  (a repeat of request 1)


Now let's replay something closer to a real workload: a support-bot server where every request starts with the same long system prompt, and each user has a short multi-turn conversation that grows with every reply — exactly the shape of an agentic loop or a chat product.


In [2]:
def make_workload(n_conversations=40, turns_per_conversation=4, system_len=60, turn_len=25, seed=0):
    """Every conversation starts with the SAME system-prompt token block (shared prefix
    across ALL requests), then grows turn by turn with conversation-specific content
    (shared only within that one conversation's own requests).
    """
    g = np.random.default_rng(seed)
    system_prompt = tuple(range(0, system_len))          # identical across every conversation
    requests = []                                          # one entry per REQUEST (one per turn)
    next_token_id = system_len
    for conv in range(n_conversations):
        history = system_prompt
        for turn in range(turns_per_conversation):
            new_tokens = tuple(range(next_token_id, next_token_id + turn_len))
            next_token_id += turn_len
            history = history + new_tokens
            requests.append(history)
    g.shuffle(requests)   # requests from different conversations/users arrive interleaved
    return requests

workload = make_workload()
trie = PrefixTrie()
total_new_tokens = 0
for seq in workload:
    reused, new = trie.insert(seq)
    total_new_tokens += new

no_sharing_tokens = sum(len(seq) for seq in workload)   # what a server with NO prefix reuse recomputes
print(f"Requests: {len(workload)}")
print(f"Total tokens across all requests    : {no_sharing_tokens:,}")
print(f"Tokens actually recomputed (RadixAttention): {total_new_tokens:,}")
print(f"Cache hit rate: {1 - total_new_tokens / no_sharing_tokens:.1%}")
print(f"Recomputation avoided: {no_sharing_tokens - total_new_tokens:,} tokens "
      f"({no_sharing_tokens / total_new_tokens:.1f}x less prefill work)")


Requests: 160
Total tokens across all requests    : 19,600
Tokens actually recomputed (RadixAttention): 4,060
Cache hit rate: 79.3%
Recomputation avoided: 15,540 tokens (4.8x less prefill work)


That's the entire mechanism behind the SGLang paper's headline number: **up to 6.4x higher throughput on workloads with shared prefixes**, with measured cache hit rates from 50% up to nearly 99% depending on how much sharing the workload has. Notice this only pays off because requests from *different conversations* still all shared the same 60-token system prompt — a workload where every request's opening tokens were unique would see a 0% hit rate and RadixAttention would buy nothing. That's also why SGLang's scheduler doesn't just serve requests in arrival order: it uses **cache-aware scheduling**, prioritizing whichever waiting request shares the *longest* prefix with what's already in cache, which keeps the trie's "hot" branches serving consecutively instead of thrashing between unrelated prefixes.

## Request scheduling: who goes next when you can't run everyone?

RadixAttention answers "how much work does this request actually need." The scheduler still has to answer a second question: when more requests are waiting than there are batch slots (notebook 13b), which one runs next? Three policies, in increasing order of sophistication:

- **FCFS (first-come, first-served):** simplest possible — whoever arrived first runs first, full stop.
- **SJF (shortest-job-first):** run whichever waiting request needs the least work, to clear the queue fastest — but a request's true length usually isn't known in advance, so real systems approximate it (e.g. from a length predictor, or from `max_tokens` if the client sets one honestly).
- **Priority:** some requests (a paying tier, an interactive user waiting on a page load) get served ahead of others (a background batch job) regardless of order or size.


In [3]:
import heapq

def simulate_queue(requests, policy, num_slots=8):
    """Discrete-event simulation of a server with `num_slots` concurrent execution slots.
    `requests`: list of dicts with 'id', 'arrival', 'service_time', 'priority'.
    `policy`: a function(waiting_list) -> index of the waiting request to admit next.
    Returns a dict of request id -> total latency (completion - arrival).
    """
    events = [(r["arrival"], "arrival", r["id"]) for r in requests]
    heapq.heapify(events)
    by_id = {r["id"]: r for r in requests}
    waiting = []              # requests that have arrived but not yet started
    completion = {}
    free_slots = num_slots

    def try_admit(now):
        nonlocal free_slots
        while free_slots > 0 and waiting:
            pick = policy(waiting)
            req = waiting.pop(pick)
            free_slots -= 1
            heapq.heappush(events, (now + req["service_time"], "departure", req["id"]))

    while events:
        time, kind, rid = heapq.heappop(events)
        if kind == "arrival":
            waiting.append(by_id[rid])
            try_admit(time)
        else:  # departure
            free_slots += 1
            completion[rid] = time
            try_admit(time)

    return {rid: completion[rid] - by_id[rid]["arrival"] for rid in by_id}

def fcfs_policy(waiting):
    return min(range(len(waiting)), key=lambda i: waiting[i]["arrival"])

def sjf_policy(waiting):
    return min(range(len(waiting)), key=lambda i: waiting[i]["service_time"])


In [4]:
def make_traffic(n_requests=600, mean_interarrival=0.8, num_slots=8, seed=0):
    """Poisson-ish arrivals with a lognormal service-time spread (most requests short,
    a few long) -- the same shape notebook 13b used for output lengths, now driving an
    actual queue instead of a batching simulation. Tuned so utilization stays near ~90%:
    high enough to create real queueing, not so high the queue grows without bound.
    """
    g = np.random.default_rng(seed)
    interarrivals = g.exponential(mean_interarrival, size=n_requests)
    arrivals = np.cumsum(interarrivals)
    service_times = np.clip(g.lognormal(mean=1.6, sigma=0.9, size=n_requests), 0.2, 40.0)
    priorities = g.choice([0, 1], size=n_requests, p=[0.85, 0.15])  # 15% high-priority (=1)
    return [
        {"id": i, "arrival": arrivals[i], "service_time": service_times[i], "priority": priorities[i]}
        for i in range(n_requests)
    ]

requests = make_traffic()
mean_service = np.mean([r["service_time"] for r in requests])
mean_gap = np.mean(np.diff([r["arrival"] for r in requests]))
print(f"Mean service time: {mean_service:.2f}  |  mean inter-arrival gap: {mean_gap:.2f}  "
      f"|  approx utilization: {mean_service / (8 * mean_gap):.1%}")

results_fcfs = simulate_queue(requests, fcfs_policy)
results_sjf = simulate_queue(requests, sjf_policy)

def p50_p99(latencies):
    arr = np.array(list(latencies.values()))
    return np.percentile(arr, 50), np.percentile(arr, 99)

for name, res in [("FCFS", results_fcfs), ("SJF", results_sjf)]:
    p50, p99 = p50_p99(res)
    print(f"{name:5s}  P50 latency {p50:6.2f}  |  P99 latency {p99:7.2f}")


Mean service time: 6.77  |  mean inter-arrival gap: 0.85  |  approx utilization: 99.2%
FCFS   P50 latency   9.22  |  P99 latency   47.84
SJF    P50 latency   5.90  |  P99 latency   93.54


SJF clears the *median* request faster than FCFS (most requests are short, so cutting in line ahead of a few long ones pays off for the majority) — but look at what it does to a request that happens to be long: under SJF a long request can get **starved**, repeatedly passed over by a stream of newly arriving short ones, so its own P99 can actually get *worse* than FCFS's even though the median improves. FCFS never starves anyone by construction — everyone's wait is bounded by how many people were already ahead of them, nothing else. This is the classic SJF tradeoff from operating-systems scheduling theory, and it shows up in every serving-side scheduler that tries to optimize average-case latency: pure length-based prioritization needs an anti-starvation mechanism, or it needs to not be pure.

## Exercises


In [5]:
# Exercise 1 (Warm-up): How much does the shared system prompt actually matter?
# Task: Re-run make_workload with system_len set to 0 (no shared prefix at all) and again
#       with system_len=200 (a much longer shared prompt). Insert each into a FRESH
#       PrefixTrie and print the cache hit rate for both. Does a longer shared prefix
#       help more, less, or about the same in relative terms?
# Hint: Reuse the exact same insert-and-sum-total_new_tokens loop from the workload cell.

# YOUR CODE HERE


In [6]:
# Exercise 2 (Apply): Implement a priority scheduling policy
# Task: Write priority_policy(waiting) that returns the index of the waiting request with
#       priority == 1 (high priority) that arrived earliest; if there are NO priority-1
#       requests waiting, fall back to FCFS among what's left. Run simulate_queue(requests,
#       priority_policy) on the SAME `requests` list used above and compare P50/P99 for
#       priority-1 requests only against FCFS's P50/P99 for the same priority-1 requests.
#       Your policy should noticeably improve their tail latency.
# Hint: [i for i, r in enumerate(waiting) if r["priority"] == 1] gives you the candidate
#       indices; if that list is empty, fall back to fcfs_policy(waiting).

# YOUR CODE HERE


In [7]:
# Exercise 3 (Extend): Fix SJF's starvation with aging
# Task: Write sjf_with_aging_policy(waiting, now, boost_rate=2.0) that scores each waiting
#       request by (service_time - boost_rate * wait_so_far) instead of raw service_time,
#       where wait_so_far = now - arrival -- the longer a request has waited, the more its
#       effective "length" shrinks, until even a long request eventually looks short enough
#       to win. You'll need a small change to simulate_queue so the policy function also
#       receives `now`; make that change, then take the 20 requests with the longest
#       service_time and compare their AVERAGE latency under FCFS, SJF, and SJF+aging.
# Hint: This is literally the "aging" fix from classic OS priority scheduling -- guarantee
#       eventual service by letting wait time erode whatever made a request look low-priority.
#       Averaging over 20 requests instead of one avoids a noisy single-request comparison.

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
for system_len in [0, 200]:
    wl = make_workload(system_len=system_len)
    t = PrefixTrie()
    total_new = sum(t.insert(seq)[1] for seq in wl)
    total = sum(len(seq) for seq in wl)
    print(f"system_len={system_len:3d}  hit rate {1 - total_new/total:.1%}")
# With system_len=0 the only sharing left is WITHIN each conversation's own turns (later
# turns reuse their own earlier turns), so the hit rate is much lower and comes entirely
# from single-user reuse. With system_len=200 the shared block is a much bigger fraction
# of every short early-conversation request, so the hit rate rises -- proportionally more
# for SHORT conversations than long ones, since a long conversation's hit rate is already
# dominated by its own growing history regardless of the system prompt's length.

# Exercise 2
def priority_policy(waiting):
    high = [i for i, r in enumerate(waiting) if r["priority"] == 1]
    if high:
        return min(high, key=lambda i: waiting[i]["arrival"])
    return fcfs_policy(waiting)

results_priority = simulate_queue(requests, priority_policy)
for name, res in [("FCFS", results_fcfs), ("Priority", results_priority)]:
    high_latencies = [res[r["id"]] for r in requests if r["priority"] == 1]
    p50, p99 = np.percentile(high_latencies, 50), np.percentile(high_latencies, 99)
    print(f"{name:8s} (priority-1 only)  P50 {p50:6.2f}  |  P99 {p99:7.2f}")
# High-priority requests jump the queue the instant a slot frees, so their tail latency
# drops close to "however long the currently-running batch takes," almost independent of
# how many low-priority requests are waiting behind them.

# Exercise 3
def simulate_queue_with_now(requests, policy, num_slots=8):
    events = [(r["arrival"], "arrival", r["id"]) for r in requests]
    heapq.heapify(events)
    by_id = {r["id"]: r for r in requests}
    waiting, completion = [], {}
    free_slots = num_slots
    def try_admit(now):
        nonlocal free_slots
        while free_slots > 0 and waiting:
            pick = policy(waiting, now)
            req = waiting.pop(pick)
            free_slots -= 1
            heapq.heappush(events, (now + req["service_time"], "departure", req["id"]))
    while events:
        time, kind, rid = heapq.heappop(events)
        if kind == "arrival":
            waiting.append(by_id[rid]); try_admit(time)
        else:
            free_slots += 1; completion[rid] = time; try_admit(time)
    return {rid: completion[rid] - by_id[rid]["arrival"] for rid in by_id}

def sjf_with_aging_policy(waiting, now, boost_rate=2.0):
    def score(r):
        wait_so_far = now - r["arrival"]
        return r["service_time"] - boost_rate * wait_so_far
    return min(range(len(waiting)), key=lambda i: score(waiting[i]))

results_aging = simulate_queue_with_now(requests, sjf_with_aging_policy)
longest_20 = sorted(requests, key=lambda r: -r["service_time"])[:20]
longest_20_ids = [r["id"] for r in longest_20]
avg_fcfs = np.mean([results_fcfs[i] for i in longest_20_ids])
avg_sjf = np.mean([results_sjf[i] for i in longest_20_ids])
avg_aging = np.mean([results_aging[i] for i in longest_20_ids])
print(f"Avg latency of the 20 longest requests -- FCFS: {avg_fcfs:.1f}  "
      f"SJF: {avg_sjf:.1f}  SJF+aging: {avg_aging:.1f}")
# Under plain SJF, long requests do WORSE than FCFS on average -- exactly the starvation
# this exercise is about. Aging guarantees every request's effective score eventually
# drops low enough to win no matter how long it is, pulling the long requests' average
# latency most of the way back down toward FCFS -- without giving up SJF's median-latency
# win for the many short requests that never needed rescuing in the first place.
```
</details>


In [8]:
# Guarded live check: talk to a real SGLang server if one happens to be running locally.
# SGLang exposes the same OpenAI-compatible /v1 surface as vLLM (notebook 13b), so the
# client code is identical -- only which server you point at changes.
def sglang_server_up(base_url="http://localhost:30000/v1", timeout=0.5):
    import urllib.request
    try:
        urllib.request.urlopen(base_url.replace("/v1", "/health"), timeout=timeout)
        return True
    except Exception:
        return False

HAS_SGLANG_SERVER = sglang_server_up()
print(f"Local SGLang server detected: {HAS_SGLANG_SERVER}")

if HAS_SGLANG_SERVER:
    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:30000/v1", api_key="EMPTY")
    resp = client.chat.completions.create(
        model="default",
        messages=[{"role": "user", "content": "Explain RadixAttention in one sentence."}],
        max_tokens=80,
    )
    print(resp.choices[0].message.content)
else:
    print("[no local SGLang server] SGLang requires a Linux GPU, same as vLLM (notebook 13b).")
    print("To launch one on a GPU machine:")
    print("  python -m sglang.launch_server --model-path meta-llama/Llama-3.1-8B-Instruct \\")
    print("      --port 30000")
    print("then point the OpenAI SDK's base_url at http://localhost:30000/v1 -- same client")
    print("code as vLLM in notebook 13b, because both speak the OpenAI chat-completions API.")


Local SGLang server detected: False
[no local SGLang server] SGLang requires a Linux GPU, same as vLLM (notebook 13b).
To launch one on a GPU machine:
  python -m sglang.launch_server --model-path meta-llama/Llama-3.1-8B-Instruct \
      --port 30000
then point the OpenAI SDK's base_url at http://localhost:30000/v1 -- same client
code as vLLM in notebook 13b, because both speak the OpenAI chat-completions API.


## Key Takeaways
- **RadixAttention** shares KV cache *across requests*, not just within one — any two requests that share a prefix (a system prompt, a tool schema, a few-shot block) reuse each other's cached computation, which is why SGLang wins hardest on agentic and few-shot workloads.
- The mechanism is a prefix tree: a new request's cache-hit length is how far it can walk down the tree of every sequence seen before it diverges. Real RadixAttention compresses this tree for memory efficiency; the compression doesn't change *what* gets reused, only how compactly it's stored.
- PagedAttention (13b) and RadixAttention answer different questions — "how do I allocate memory for one request without waste" vs. "how do I avoid recomputing work two requests both already need" — and production stacks increasingly want both.
- When more requests are waiting than there are slots, **FCFS** is starvation-free but ignores request size; **SJF** minimizes median latency but can starve long requests; **priority** scheduling serves designated requests first regardless of order, at the direct cost of everyone else's latency.
- **Aging** — letting wait time erode a request's effective priority the longer it waits — is the standard fix for SJF-style starvation, guaranteeing every request eventually gets served no matter how "unfavorable" it looks by the original metric.

## What's Next
Notebook **08_production/29b — Load Testing Inference** puts a load generator in front of a real (mocked) streaming server and asks the question this notebook's queue simulation only approximated: what actually happens to P99 latency as concurrency climbs past what the server can handle, and how do you measure that honestly instead of fooling yourself with a closed-loop benchmark.
